# TrustLens multi-generator AI detector retraining

This notebook streams Tiny-GenImage and does not download the full dataset. It trains only on the dataset's `train` split. The existing 270-image TrustLens compact pack remains untouched for the final promotion test. The output is a candidate checkpoint and never overwrites the deployed model automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Change this only if your project uses a different Drive folder.
BACKEND_DIR = Path('/content/drive/MyDrive/TrustLens/trustlens_MJ-1/backend')
TRAINING_SCRIPT = BACKEND_DIR / 'scripts' / 'train_ai_detector_streaming.py'
assert TRAINING_SCRIPT.is_file(), f'Training script not found: {TRAINING_SCRIPT}'
print('Backend:', BACKEND_DIR)

In [ ]:
%pip install -q "datasets>=4,<5" "scikit-learn>=1.5,<2" "tqdm>=4.66"

In [ ]:
import subprocess
import sys

OUTPUT_DIR = BACKEND_DIR / 'training_outputs'
CHECKPOINT = OUTPUT_DIR / 'ai_detector_multigen_candidate.pth'
REPORT = OUTPUT_DIR / 'ai_detector_multigen_report.json'

command = [
    sys.executable, str(TRAINING_SCRIPT),
    '--output', str(CHECKPOINT),
    '--report', str(REPORT),
    '--train-samples', '8000',
    '--validation-samples', '1200',
    '--epochs', '8',
    '--batch-size', '32',
]
subprocess.run(command, cwd=BACKEND_DIR, check=True)
print('Candidate:', CHECKPOINT)
print('Report:', REPORT)

In [ ]:
import json

report = json.loads(REPORT.read_text())
print('Best validation AUROC:', round(report['best_validation_auroc'], 4))
print('Promotion status:', report['promotion_status'])

## After training

Copy only the candidate `.pth` and its JSON report back to the local project. Do not rename it to `ai_detector_best.pth` yet. Run the TrustLens compact calibration workflow first; promote the candidate only if all activation gates pass.